In [1]:
import requests
import json
from typing import List, Optional, Dict, Any
from pydantic import BaseModel
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware

In [2]:
app = FastAPI(title="Jarvis Assistant API")

# Add CORS middleware to allow web clients to connect
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # For development; restrict in production
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

In [ ]:
# Ollama API configuration
OLLAMA_BASE_URL = "http://localhost:11434"
MODEL_NAME = "vanilj/phi-4-unsloth:latest"  # Update with your Ollama model name

In [6]:
# Pydantic models for request/response validation
class Message(BaseModel):
    role: str
    content: str

class ChatRequest(BaseModel):
    messages: List[Message]
    temperature: Optional[float] = 0.7
    max_tokens: Optional[int] = 1000
    stream: Optional[bool] = False

class ChatResponse(BaseModel):
    response: str
    model: str
    
class HealthResponse(BaseModel):
    status: str
    model: str

In [7]:
@app.get("/health", response_model = HealthResponse)
async def health_check():
    """Check if the API and Ollama service are running."""
    try:
        # Check if Ollama is accessible
        response = requests.get(f"{OLLAMA_BASE_URL}/api/tags")
        if response.status_code == 200:
            return {"Status":"ok", "model":MODEL_NAME}
        else:
            raise HTTPException(status_code=503, detail="Ollama service unavailable")
    except Exception as e:
        raise HTTPException(status_code=503, detail= f"Error connecting to Ollama: {str(e)}")
    

In [ ]:
@app.post("/chat", response_model=ChatResponse)
async def chat(request: ChatRequest):
    """Process a chat request through Ollama"""
    try:
        # Format messages for Ollama API
        # Convert message format to Ollama's expected format
        prompt = ""
        for msg in request.messages:
            if msg.role == "user":
                prompt += f"User: {msg.content}"
            elif msg.role == "assistant":
                prompt += f"Assistant: {msg.content}"
            elif msg.role == "system":
                prompt += f"System: {msg.content}\n"
        
        # Make request to Ollama
        ollama_request = {
            "model": MODEL_NAME,
            "prompt": prompt,
            "temperature": request.temperature,
            "num_preduct": request.max_tokens,
            "stream": False
        }